# 01 — Data Loading and Exploration

**Goals**
- Load all raw CSVs from `data/raw/` (one per cycle)
- Basic sanity checks: correct columns present, no wildly out-of-range values
- Plot each cycle's raw moisture-over-time curve
- Print cycle count per condition (indoor vs outdoor)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = Path('../data/raw')          # relative to notebooks/
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLS = [
    'timestamp', 'temperature_C', 'humidity_pct', 'soil_moisture_pct',
    'light_lux', 'condition', 'cycle_id'
]


In [ ]:
# Load every CSV in data/raw/
cycles = {}
for f in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(f, parse_dates=['timestamp'])
    cycle_id = df['cycle_id'].iloc[0]
    cycles[cycle_id] = df
    print(f'Loaded {f.name:30s} → {len(df):3d} rows  cycle_id={cycle_id}')

print(f'\nTotal cycles loaded: {len(cycles)}')


In [ ]:
# Column presence check
print('=== Column checks ===')
for cid, df in cycles.items():
    missing = set(REQUIRED_COLS) - set(df.columns)
    extra   = set(df.columns) - set(REQUIRED_COLS)
    print(f'{cid}: missing={missing or "none"}, extra={extra or "none"}')


In [ ]:
# Value-range sanity checks
print('=== Value range checks ===')
for cid, df in cycles.items():
    issues = []
    if (df['humidity_pct'] > 100).any() or (df['humidity_pct'] < 0).any():
        issues.append('humidity out of [0,100]')
    if (df['soil_moisture_pct'] < 0).any() or (df['soil_moisture_pct'] > 100).any():
        issues.append('soil_moisture out of [0,100]')
    if (df['temperature_C'] < -20).any() or (df['temperature_C'] > 60).any():
        issues.append('temperature extreme')
    if (df['light_lux'] < 0).any():
        issues.append('negative light')
    status = issues or ['OK']
    print(f'{cid}: {status} | moisture [{df.soil_moisture_pct.min():.1f}, {df.soil_moisture_pct.max():.1f}]')


In [ ]:
# Plot moisture-over-time for every cycle
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.flatten()
for ax, (cid, df) in zip(axes, cycles.items()):
    ax.plot(df['timestamp'], df['soil_moisture_pct'], marker='o', markersize=3)
    ax.set_title(f'{cid} ({df.condition.iloc[0]})')
    ax.set_ylabel('Soil Moisture %')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / '01_moisture_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → results/01_moisture_curves.png')


In [ ]:
# Cycle count per condition
cond_counts = pd.Series({cid: df.condition.iloc[0] for cid, df in cycles.items()}).value_counts()
print('Cycles per condition:')
print(cond_counts)
print(f'\nIndoor: {cond_counts.get("indoor", 0)}   Outdoor: {cond_counts.get("outdoor", 0)}')
